In [0]:
dbutils.widgets.text("2. perfume_id_input", "")
perfume_id_input = dbutils.widgets.get("2. perfume_id_input").strip()
if perfume_id_input:  # This checks if the string is non-empty
    perfume_id_input = int(perfume_id_input)
else:
    perfume_id_input = None
print(perfume_id_input)

dbutils.widgets.text("9. number of top results", "")
limit_n = int(dbutils.widgets.get("9. number of top results").strip())
print(limit_n)

None
10


In [0]:
from rapidfuzz import fuzz
from pyspark.sql.functions import col, lit, pandas_udf
from pyspark.sql.types import DoubleType
import pandas as pd

dbutils.widgets.text("1. search_term", "")
search_term = dbutils.widgets.get("1. search_term").strip()  # Fixed: strip() on the result, not the key
print(f"Searching for \"{search_term}\"...")

# Load your data as Spark DataFrame
df = spark.table("fragrance_db.default.fragrance_cleaned")

# Define pandas UDF for scoring
@pandas_udf(DoubleType())
def smart_fuzzy_score(names: pd.Series, search_term_series: pd.Series) -> pd.Series:
    search_term = search_term_series.iloc[0].lower()
    
    def calculate_score(name):
        if pd.isna(name):
            return 0.0
            
        name_lower = str(name).lower()
        
        # Multiple scoring strategies
        ratio = fuzz.ratio(name_lower, search_term)
        token_set = fuzz.token_set_ratio(name_lower, search_term)
        token_sort = fuzz.token_sort_ratio(name_lower, search_term)
        
        # Bonus for substring match
        substring_bonus = 20 if search_term in name_lower else 0
        
        # Weighted combination
        final_score = (
            ratio * 0.4 +
            token_set * 0.4 +
            token_sort * 0.2 +
            substring_bonus * 0.1
        )
        
        return final_score
    
    return names.apply(calculate_score)

# Apply the search
search_results = (df
    .withColumn("score", smart_fuzzy_score(col("name"), lit(search_term)))
    .filter(col("score") > 70)
    .orderBy(col("score").desc())
    .limit(20)
)

# Display results
display(search_results.select("id", "name", "brand", "score"))

top_search_result = search_results.first().id
print(top_search_result)

Searching for "herbes tourbulantes"...


id,name,brand,score
69275,Herbes Troublantes,Guerlain,91.8918918918919


69275


In [0]:
import numpy as np
from pyspark.sql.functions import udf, col, lit
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Cosine similarity function
def cosine_sim(a, b):
    a, b = np.array(a, dtype=np.float32), np.array(b, dtype=np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

cosine_udf = udf(cosine_sim, DoubleType())

# Get the vector of the target perfume
if perfume_id_input is not None:
    top_search_result = perfume_id_input
else:
    pass

target_vec = spark.table("fragrance_db.default.fragrance_embeddings") \
    .filter(col("id") == top_search_result) \
    .select("embedding") \
    .collect()[0][0]

target_name = spark.table("fragrance_db.default.fragrance_cleaned") \
    .filter(col("id") == top_search_result) \
    .select("name") \
    .collect()[0][0]

# Compute similarity
df = spark.table("fragrance_db.default.fragrance_embeddings")
df_result = df.withColumn(
    "similarity", 
    cosine_udf(col("embedding"), lit(target_vec))
).withColumn(
    "rank", 
    row_number().over(Window.orderBy(col("similarity").desc()))
)

limit_n = limit_n + 1 # To get top n results excluding selected perfume
df_top_similarity_results = df_result.orderBy(col("rank").asc()).limit(limit_n)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(df_top_similarity_results)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id perfume_string embedding partition_key similarity rank 69275 accords_green accords_citrus accords_musky accords_white_floral accords_powdery accords_fresh_spicy accords_sweet accords_animalic List(0.051026043, 0.005466139, 0.04344376, 0.0604514, 0.009498423, 0.09493733, 0.0274301, -0.01744975, 0.041380484, 0.011852468, 0.048146196, -0.07130826, 0.025033107, -0.04901297, 0.03073076, 0.060605086, -0.0013006241, 0.0020451723, -0.108237945, -0.046970136, -0.0343306, 0.013006068, 0.0026378103, 0.03566349, -0.019884061, 0.06811867, -0.02134866, 0.05073119, 0.02887321, -0.08387126, -0.021561693, 0.03255268, 0.051359612, 7.123E-4, 0.010606814, -0.072777115, -0.08213742, -0.01721911, 0.071796104, 0.027067864, 0.018687459, -0.08109734, 0.0076400777, 0.026003463, -0.019870086, 0.014767747, -0.023247333, 0.008581974, -0.014266603, -0.0029833324, -0.002466972, -0.06526916, -0.09395673, 0.06475322, -0.041609712, -0.055809457, -0.08056078, 0.068366624, -0.0113906115, -0.017517986, 0.007228879, -0.040007345, 0.025201794, 0.03341411, -0.10762691, 0.060478766, -0.07995409, 0.0031588417, 0.0056647556, 0.021447532, -0.029446622, 0.07221055, 0.03773241, 0.03881579, -0.035213243, 0.031247405, 0.03678832, -0.010548425, -0.055337757, -0.103713244, -0.03494682, 0.08402153, -0.05549453, -0.06410926, -0.017706897, 0.07075483, -0.06276929, -0.009978285, -0.09618574, -0.030063901, -0.057103015, -0.09083102, 0.17444046, -0.052643534, -0.010015003, 0.08034228, 0.035837494, -0.07579215, -0.0070101046, 0.0031755755, 0.054306835, 0.03514428, -0.027237054, -0.031170985, -0.12518726, 0.023784671, -0.07230762, -0.055686407, 0.005692818, 0.016394563, -0.050059117, -9.443191E-4, -0.00705404, -0.11219315, -0.05594479, -0.062800944, 0.013828022, -0.014752431, 0.100237526, -0.0062350202, -0.005867524, -0.092072174, -0.010419599, -0.021740835, -0.073230356, -0.012644167, 0.022782944, 1.7144245E-33, -0.007392723, 0.010813341, 0.010789892, 0.010138318, 0.061997663, -0.096471824, 0.001050965, 0.014813888, -0.079847455, 0.0479877, -0.013063743, 0.05522499, -0.040944833, -0.028631702, 0.055659752, 0.054693237, -0.0059964913, 0.019998163, -0.03049865, -0.061170958, -0.05653066, 0.05977939, 0.04910426, 0.06587194, 0.019574717, -0.05584531, 0.040035374, -0.0390799, -0.042189524, -0.030971082, 0.004148946, 0.06987491, -0.03425676, 0.06489884, -0.09731617, -0.024129543, -0.06186106, -0.041646346, 0.010319335, -0.003709467, 0.022008516, -0.014564801, -0.023580909, 0.1230301, 0.09593037, 0.08423184, -0.04027785, 0.05128777, 0.082684666, -0.023885356, -0.02456874, -7.846106E-4, 0.025064986, 0.009262751, 0.017301802, -0.051698573, 0.020586567, -0.008003082, -0.053859215, 0.0060153734, -0.021367384, 0.04480626, -0.018944943, -0.02668559, -0.058899283, 0.012505668, -0.08665041, -0.009887125, 0.029805304, -0.027208064, 0.009249607, -0.032832295, -0.031171512, 0.062190004, 0.08115097, 0.0074301297, 0.07541119, -0.097502165, 0.031478677, -0.056060996, -0.05124345, -0.025979308, -0.05326746, 0.09823243, -0.048342522, -0.0051128482, -0.049984995, -0.039783753, 0.03096439, 0.017962346, -0.08605863, 0.024329413, 0.048970964, 4.819689E-4, -0.081560545, -3.0260534E-33, 0.042774793, 0.0016441197, 2.5509804E-5, 0.08562561, 0.09755791, -0.038498364, -0.036337964, -0.009418679, 0.038253658, 0.013916987, 0.08637073, -0.026201662, -0.011490832, -0.030266766, 0.01307723, 0.029795423, 0.043673106, 0.10586264, 0.058542714, 0.058917623, -0.08300561, 0.025951646, -0.030086985, 0.04239987, -0.024155669, 0.09729346, 0.017609948, -0.05818623, -0.002343262, 0.036800757, -0.003602616, -0.047069434, -0.044292454, 0.035807703, -0.01731595, -0.0888131, 0.028038979, -0.061604884, -0.10708239, 0.038460884, -0.05309291, 0.010356227, 0.062265985, 0.14794625, -0.008928119, -0.09060107, -0.06388455, -0.02360333, -0.058948353, -0.0017115761, 0.06461516, -0.026326004, -0.06276216, 0.052249946, -0.0030292836, 0.10828636, -0.001968667, 0.004952643, -0.01120583, -0.0041921246, -0.022136794, 0.018259846, -0.0077785025, 0

In [0]:
df_frag = spark.sql(f"""
    SELECT id, name, brand, gender, accords, top_notes, mid_notes, base_notes, url
    FROM fragrance_db.default.fragrance_cleaned
    """
)

df_result_view = df_frag.join(
    df_top_similarity_results.select("rank", "id", "similarity"),
    on="id",
    how="inner"
).select((df_top_similarity_results["rank"]-1).alias("rank"), df_frag["*"], df_top_similarity_results["similarity"]
).orderBy(col("similarity").desc())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
count = df_result_view.count() - 1

print(f"Top {count} similar perfumes to {target_name} below:")
display(df_result_view.offset(1))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Top 10 similar perfumes to Herbes Troublantes below:


rank,id,name,brand,gender,accords,top_notes,mid_notes,base_notes,url,similarity
1,59017,N 19v,InEstasy,women,"fruity, amber, white floral, citrus, musky, powdery, fresh spicy, sweet, green, animalic",,,,https://www.fragrantica.com/perfume/InEstasy/N-19v-59017.html,0.9926376938819885
2,8964,Hanae,Keiko Mecheri,women,"citrus, white floral, fruity, musky, sweet, fresh, powdery, fresh spicy, animalic, green",,,,https://www.fragrantica.com/perfume/Keiko-Mecheri/Hanae-8964.html,0.992536187171936
3,5370,Snowpeach,Renee,women,"fruity, green, citrus, white floral, fresh spicy, sweet, floral, musky, powdery, animalic",,,,https://www.fragrantica.com/perfume/Renee/Snowpeach-5370.html,0.9903098940849304
4,32868,Slezy angela,YanFroloff,women,"citrus, floral, tuberose, musky, fruity, white floral, fresh spicy, powdery, green, animalic",,,,https://www.fragrantica.com/perfume/YanFroloff/Slezy-angela-32868.html,0.9887061715126038
5,50827,SP Cologne,SP Parfums Sven Pritzkoleit,unisex,"white floral, musky, tuberose, powdery, woody, fresh, warm spicy, green, animalic, citrus",,,,https://www.fragrantica.com/perfume/SP-Parfums-Sven-Pritzkoleit/SP-Cologne-50827.html,0.9880797266960144
6,74878,Osmanthus,FUMparFUM,unisex,"floral, amber, musky, citrus, fruity, fresh, powdery, soft spicy, green, animalic",,,,https://www.fragrantica.com/perfume/FUMparFUM/Osmanthus-74878.html,0.9878197312355042
7,51482,Vines,Regime des Fleurs,unisex,"green, white floral, citrus, fruity, musky, fresh, fresh spicy, sweet",,,,https://www.fragrantica.com/perfume/Regime-des-Fleurs/Vines-51482.html,0.9872162938117981
8,69584,Convalia,FUMparFUM,unisex,"fresh, floral, citrus, amber, white floral, musky, green, powdery, fresh spicy, animalic",,,,https://www.fragrantica.com/perfume/FUMparFUM/Convalia-69584.html,0.9864056706428528
9,19168,Lys,Molinard,women,"white floral, citrus, vanilla, musky, powdery, fresh, soft spicy, animalic, sweet",,,,https://www.fragrantica.com/perfume/Molinard/Lys-19168.html,0.9858610033988953
10,2490,Benetton Pure Sport Women,Benetton,women,"white floral, musky, citrus, soft spicy, powdery, fresh, fresh spicy, floral, green, sweet",,,,https://www.fragrantica.com/perfume/Benetton/Benetton-Pure-Sport-Women-2490.html,0.985803484916687
